### This notebook format data for training deepSCENIC from a SCENIC+ object

In [1]:
import pickle
import scanpy as sc
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

Read the SCENIC+ object

In [2]:
import dill
scplus_folder = "/lustre1/project/stg_00002/lcb/gpartel/projects/deepSCENIC/data/DPCL/"
scenicpls_obj = dill.load(
        open(os.path.join(scplus_folder, 'scplus_obj.pkl'), 'rb'))

/staging/leuven/stg_00002/lcb/gpartel/software/anaconda3/envs/DeepSEM/lib/python3.8/site-packages/requests/__init__.py:102: RequestsDependencyWarning: urllib3 (1.26.9) or chardet (5.1.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({})/charset_normalizer ({}) doesn't match a supported "
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [3]:
# PATH WHERE TO SAVE THE DATA
DATA_PATH = '/staging/leuven/stg_00002/lcb/gpartel/projects/deepSCENIC/data/DPCL/'

# Save cell metadata
scenicpls_obj.metadata_cell.to_csv(DATA_PATH+'cells_metadata.csv')

Extract RNA and ATAC data, preprocessing and filtering

In [4]:
ad_rna = sc.AnnData(scenicpls_obj.to_df('EXP'))
ad_rna

AnnData object with n_obs × n_vars = 4000 × 25651

In [5]:
sc.pp.filter_genes(ad_rna, min_cells=10)
ad_rna

AnnData object with n_obs × n_vars = 4000 × 25651
    var: 'n_cells'

In [6]:
sc.pp.normalize_total(ad_rna)
sc.pp.log1p(ad_rna)
ad_rna.layers['log_norm'] = ad_rna.X

In [7]:
# Filter regions based on minimum number of fragments
min_nr_acc = 20
scenicpls_obj.subset(regions = scenicpls_obj.metadata_regions[scenicpls_obj.metadata_regions.cisTopic_nr_acc>min_nr_acc].index.to_list())

In [8]:
ad_atac = sc.AnnData(scenicpls_obj.to_df('ACC').T)
ad_atac

/tmp/ipykernel_3697971/3854231756.py:1: FutureWarning: X.dtype being converted to np.float32 from int32. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  ad_atac = sc.AnnData(scenicpls_obj.to_df('ACC').T)


AnnData object with n_obs × n_vars = 4000 × 398953

Get regions per gene (up to 150 kb up and down stream of the gene)

In [9]:
from pycisTopic.utils import region_names_to_coordinates
from scenicplus.enhancer_to_gene import get_search_space
import pyranges as pr

max_dist = 150000
search_space = get_search_space(scenicpls_obj, pr_annot = pr.PyRanges(region_names_to_coordinates(scenicpls_obj.region_names.values)),
                                species = 'hsapiens',
                                assembly = 'hg38',
                                upstream = [1000, max_dist],
                                downstream = [1000, max_dist],
                                inplace=False,
                               )

2023-09-14 13:02:41,987 R2G          INFO     Downloading gene annotation from biomart dataset: hsapiens_gene_ensembl
2023-09-14 13:02:56,311 R2G          INFO     Downloading chromosome sizes from: http://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.chrom.sizes
2023-09-14 13:02:57,600 R2G          INFO     Extending promoter annotation to 10 bp upstream and 10 downstream
2023-09-14 13:03:00,347 R2G          INFO     Extending search space to:
            						150000 bp downstream of the end of the gene.
            						150000 bp upstream of the start of the gene.
2023-09-14 13:03:17,139 R2G          INFO     Intersecting with regions.


join: Strand data from other will be added as strand data to self.
If this is undesired use the flag apply_strand_suffix=False.
To turn off the warning set apply_strand_suffix to True or False.


2023-09-14 13:03:18,557 R2G          INFO     Calculating distances from region to gene
2023-09-14 13:05:41,462 R2G          INFO     Imploding multiple entries per region and gene
2023-09-14 13:11:51,146 R2G          INFO     Done!


Building region to gene distance matrix

In [10]:
search_space['Abs_distance'] = np.abs([x[0] for x in search_space['Distance'].to_list()]).astype(int)

In [11]:
search_space_subset = search_space.loc[[gene in ad_rna.var_names for gene in search_space['Gene']]]
regions = search_space_subset.loc[:, 'Name'].unique().tolist()

In [12]:
scenicpls_obj.subset(regions = regions)

In [13]:
r2gdist = search_space_subset.pivot(index='Name', columns='Gene', values='Abs_distance')

/tmp/ipykernel_3697971/2169892100.py:1: PerformanceWarning: The following operation may generate 5588481668 cells in the resulting pandas object.
  r2gdist = search_space_subset.pivot(index='Name', columns='Gene', values='Abs_distance')


In [14]:
# Scale matrix between 0 and 1
r2gdist = r2gdist / max_dist
r2gdist = r2gdist.fillna(0)

In [15]:
# Ensure same order of indices
regions = r2gdist.index.values.tolist()
genes = r2gdist.columns.values.tolist()

ad_atac = ad_atac[:, regions].copy()
ad_rna = ad_rna[:, genes].copy()

Make region to gene matrix sparse

In [16]:
from scipy.sparse import coo_array, save_npz
r2gdist_coo = coo_array(r2gdist)

Import list of TFs (It can be downloaded from SCENIC+ repo: https://github.com/aertslab/scenicplus/tree/main/resources)

In [17]:
TFs = pd.read_csv("/staging/leuven/stg_00002/lcb/gpartel/software/DeepSEM_multimodal/resources/allTFs_hg38.txt", header=None)[0].to_list()

In [18]:
# Subset TFs present in RNA data
TFs = ad_rna.var_names[ad_rna.var_names.isin(TFs)]

Saving data

In [ ]:
ad_rna.write(DATA_PATH+'raw_exprMat.h5ad')
ad_atac.write(DATA_PATH+'fragment_matrix.h5ad')
save_npz(DATA_PATH+'r2gdist.npz', r2gdist_coo)
with open(DATA_PATH+'TFs.txt', 'w') as f:
    for item in TFs:
        f.write("%s\n" % item)

### Extract motif PPMs (optional)

In [2]:
# Load motif annotations

from pycistarget.utils import load_motif_annotations
annot = load_motif_annotations('homo_sapiens',  fname='/staging/leuven/stg_00002/lcb/cbravo/cluster_motif_collection_V10_no_desso_no_factorbook/snapshots/motifs-v10nr_clust-nr.hgnc-m0.00001-o0.0.tbl')
# annot = load_motif_annotations('drosophila_melanogaster')

In [3]:
# Load PWMs
import glob
# MOTIF_PATH = '/staging/leuven/stg_00002/lcb/icistarget/data/motif2tf_project/motif_collections_data/cluster_buster/all/'
MOTIF_PATH = '/staging/leuven/stg_00002/lcb/cbravo/cluster_motif_collection_V10_no_desso_no_factorbook/cluster_buster'
file_paths = glob.glob(os.path.join(MOTIF_PATH, "*.cb"))
len(file_paths)

12935

In [4]:
# Create motif-TF annotation dataframe
def parse_pwm_file(file_path, pwm_dict, metacluster_dict):
    with open(file_path, 'r') as file:
        lines = file.readlines()
        i = 0
        while i < len(lines):
            line = lines[i].strip()
            if line.startswith('>'):
                motif_name = line[1:]
                pwm = []
                i += 1
                while i < len(lines) and not lines[i].startswith('>'):
                    pwm_line = lines[i].strip().split()
                    pwm.append([float(val) for val in pwm_line])
                    i += 1
                pwm_dict[motif_name] = pwm
                metacluster_dict[motif_name] = os.path.splitext(os.path.basename(file_path))[0]
            else:
                i += 1
    return pwm_dict, metacluster_dict

pwm_dict = {}
metacluster_dict = {}
for f in tqdm(file_paths):
    pwm_dict, metacluster_dict = parse_pwm_file(f, pwm_dict, metacluster_dict)

annot = annot.loc[:,['Direct_annot', 'Orthology_annot']].dropna(how='all')
annot

  8%|▊         | 972/12935 [01:02<12:54, 15.44it/s]

KeyboardInterrupt



In [ ]:
# Save annotation dataframe
DATA_PATH = '/staging/leuven/stg_00002/lcb/gpartel/projects/deepSCENIC/data/'

annot.to_pickle(DATA_PATH + 'motif_ann_public.pkl')

In [ ]:
# Create dictionary of motif probability distributions
import torch
from torch.distributions import Categorical

PPMs_dist = {}
annot_dict = {}
for motif in tqdm(pwm_dict.keys()):
    meta_motif = metacluster_dict[motif]
    if meta_motif in annot.index:
        pwm = np.array(pwm_dict[motif])
        ppm = pwm / pwm.sum(1)[:,None]
        
        PPMs_dist[motif] = Categorical(probs=torch.tensor(ppm))
        TFs = list(set(list(annot.loc[meta_motif, :].dropna().values)))
        TFs = np.unique([tf for sublist in TFs for tf in sublist.split(', ')])
        annot_dict[motif] = TFs

len(annot_dict.keys())

# Save motif probability distributions
with open(DATA_PATH + 'PPMs_public.pkl', 'wb') as f:
    pickle.dump({'dists': PPMs_dist, 'annot':annot_dict}, f)

In [6]:
with open('/staging/leuven/stg_00002/lcb/gpartel/projects/deepSCENIC/data/PPMs_public.pkl', 'rb') as f:
            PPMs = pickle.load(f)